# 第 8 周 - 笔记本 1：设置 RAG 系统

## 目标
为 RAG 设置包含 800k 产品的 ChromaDB 矢量数据库：
1. 从 HuggingFace Hub 加载数据集
2. 使用 SentenceTransformer 创建嵌入
3. 存储在 ChromaDB 中
4. 测试相似度搜索

## 时间：20-30分钟（嵌入需要时间）

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

from src.utils.items import Item
from src.config import config

# 负载环境
# Load environment
load_dotenv()

print("✅ Environment loaded")

## 配置

In [ ]:
config.display()

## 步骤 1：加载数据集

In [ ]:
print(f"Loading dataset: {config.DATASET_NAME}")
train, val, test = Item.from_hub(config.DATASET_NAME)
all_items = train + val + test

print(f"\n✅ Loaded:")
print(f"   Training: {len(train):,} items")
print(f"   Validation: {len(val):,} items")
print(f"   Test: {len(test):,} items")
print(f"   Total: {len(all_items):,} items")

## 步骤 2：初始化 ChromaDB

In [ ]:
# 创建 ChromaDB 客户端
# Create ChromaDB client
chroma_client = chromadb.PersistentClient(path="../data/chroma")

# 创建或获取集合
# Create or get collection
collection_name = "products"

try:
    # 尝试删除现有集合
    # Try to delete existing collection
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print(f"No existing collection to delete")

# 创建新集合
# Create new collection
collection = chroma_client.create_collection(name=collection_name)
print(f"✅ Created ChromaDB collection: {collection_name}")

## 步骤 3：加载嵌入模型

In [ ]:
# 加载SentenceTransformer模型
# Load SentenceTransformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Loaded embedding model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 步骤 4：创建嵌入并存储

对于 800k 个项目，这将需要 20-30 分钟

In [ ]:
# 批量处理
# Process in batches
BATCH_SIZE = 1000
total_batches = (len(all_items) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processing {len(all_items):,} items in {total_batches} batches...")
print("This will take 20-30 minutes...\n")

for i in tqdm(range(0, len(all_items), BATCH_SIZE), desc="Embedding batches"):
    batch = all_items[i:i+BATCH_SIZE]
    
    # 准备数据
    # Prepare data
    ids = [str(item.id) for item in batch]
    documents = [item.prompt or item.summary or item.title for item in batch]
    metadatas = [
        {
            "title": item.title,
            "category": item.category,
            "price": item.price,
        }
        for item in batch
    ]
    
    # 创建嵌入
    # Create embeddings
    embeddings = model.encode(documents).astype(float).tolist()
    
    # 添加到 ChromaDB
    # Add to ChromaDB
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )

print(f"\n✅ Added {len(all_items):,} items to ChromaDB")
print(f"   Collection count: {collection.count():,}")

## 步骤 5：测试相似性搜索

In [ ]:
# 测试查询
# Test query
test_query = "wireless bluetooth headphones with noise cancellation"

print(f"Test query: {test_query}\n")

# 为查询创建嵌入
# Create embedding for query
query_embedding = model.encode([test_query]).astype(float).tolist()

# 搜索
# Search
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print("Top 5 similar products:\n")
for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
    print(f"{i}. {metadata['title']}")
    print(f"   Category: {metadata['category']}")
    print(f"   Price: ${metadata['price']:.2f}")
    print(f"   Description: {doc[:100]}...\n")

## 概括

✅ RAG 系统设置完成！

**我们做了什么：**
1. 从 HuggingFace Hub 加载 800k 产品
2. 使用 SentenceTransformer 创建嵌入
3.存储在ChromaDB矢量数据库中
4.经过测试的相似性搜索

**ChromaDB 位置：** `../data/chroma/`

**下一步：** `02_test_agents.ipynb` - 测试各个代理